# 基于MindNLP的RoBERTa模型Prompt Tuning

## 案例介绍

本案例对roberta-large模型基于GLUE基准数据集进行prompt tuning。

## 模型介绍

RoBERTa 的全称是 Robustly optimized BERT approach，可以理解为“经过更精细优化的 BERT 模型”。它由 Facebook AI（现 Meta AI）在 2019 年发布，是对 Google 的 BERT 模型的一次重大改进和重新设计。

其核心思想是：BERT 的原始设计很好，但训练不充分、配置可以优化。通过一系列改进，RoBERTa 在多个自然语言理解基准测试上超越了 BERT，成为了当时最强大的预训练模型之一。

## 环境配置

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10    | 2.7.0       | 0.5.1           |

如果你在如[昇思大模型平台](https://xihe.mindspore.cn/training-projects)、[华为云ModelArts](https://www.huaweicloud.com/product/modelarts.html)、[启智社区](https://openi.pcl.ac.cn/)等算力平台的Jupyter在线编程环境中运行本案例，可取消如下代码的注释，进行依赖库安装：

In [ ]:
# !pip install mindspore==2.7.0 mindnlp==0.5.1

其他场景可参考[MindSpore安装指南](https://www.mindspore.cn/install)与[MindSpore NLP安装指南](https://github.com/mindspore-lab/mindnlp?tab=readme-ov-file#installation)进行环境搭建。

## 数据加载与预处理

### 数据集加载

In [2]:
import argparse
import os

import mindnlp
import mindspore
import mindnlp
from tqdm import tqdm
import evaluate
from datasets import load_dataset

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from peft import (
    get_peft_config,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
    PeftType,
    PromptTuningConfig,
)

/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.10.14/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.10.14/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress 

In [ ]:
mindspore.set_context(pynative_synchronize=True)  #开启同步，方便定位

[WARNING] ME(67332:281473403209280,MainProcess):2025-11-25-17:06:03.983.000 [mindspore/context.py:1412] For 'context.set_context', the parameter 'pynative_synchronize' will be deprecated and removed in a future version. Please use the api mindspore.runtime.launch_blocking() instead.


In [ ]:
batch_size = 32
model_name_or_path = "FacebookAI/roberta-large"
task = "mrpc"
peft_type = PeftType.PROMPT_TUNING
num_epochs = 1

prompt tuning配置，任务类型选为"SEQ_CLS", 即序列分类。

In [5]:
# peft config
peft_config = PromptTuningConfig(task_type="SEQ_CLS", num_virtual_tokens=10)
# learning rate
lr = 1e-3

加载tokenizer。如模型为GPT、OPT或BLOOM类模型，从序列左侧添加padding，其他情况下从序列右侧添加padding。

In [ ]:
# load tokenizer
if any(k in model_name_or_path for k in ("gpt", "opt", "bloom")):
    padding_side = "left"
else:
    padding_side = "right"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, padding_side=padding_side)
if getattr(tokenizer, "pad_token_id") is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [7]:
tokenizer.padding_side, tokenizer.pad_token_id

('right', 1)

In [8]:
datasets = load_dataset("glue", task)

# 查看数据集的划分
print(datasets) 
train_dataset = datasets['train']

# Set the dataset format for PyTorch
train_dataset.set_format(type='torch')

# Simply iterate directly
print(train_dataset[0])

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})
{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', 'label': Tensor(shape=[], dtype=Int64, value= 1), 'idx': Tensor(shape=[], dtype=Int64, value= 0)}


### 数据集处理

In [12]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

def get_dataset(dataset, tokenizer):
    def tokenize_function(examples):
        return tokenizer(
            examples['sentence1'],
            examples['sentence2'],
            truncation=True,
            max_length=None,
        )
    
    # 应用tokenize函数
    dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=['sentence1', 'sentence2', 'idx']
    )
    
    return dataset

# 处理数据集
train_dataset = get_dataset(datasets['train'], tokenizer)
eval_dataset = get_dataset(datasets['validation'], tokenizer)

# 创建数据整理器
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 创建DataLoader
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    collate_fn=data_collator,
    shuffle=True
)

eval_dataloader = DataLoader(
    eval_dataset, 
    batch_size=batch_size, 
    collate_fn=data_collator
)

Map: 100%|██████████| 408/408 [00:00<00:00, 10199.22 examples/s]


In [13]:
train_dataloader

### 查看数据集信息

In [14]:
# 获取一个批次
batch = next(iter(train_dataloader))

print("输入ID:", batch['input_ids'].shape)
print("注意力掩码:", batch['attention_mask'].shape)
if 'labels' in batch:
    print("标签:", batch['labels'].shape)

[MS_ALLOC_CONF]Runtime config:  enable_vmm:True  vmm_align_size:2MB
输入ID: mindtorch.Size([32, 75])
注意力掩码: mindtorch.Size([32, 75])
标签: mindtorch.Size([32])


## 加载评估指标

In [ ]:
!pip install scikit-learn #安装依赖

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple


In [16]:
metric = evaluate.load("glue", task)

## 模型加载

加载模型并打印微调参数量，可以看到仅有不到0.6%的参数参与了微调。

如出现如下告警请忽略，并不影响模型的微调。

```text
The following parameters in checkpoint files are not loaded:
['lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'roberta.embeddings.position_ids']
The following parameters in models are missing parameter:
['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
```

In [17]:
# load model
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, return_dict=True)
model = get_peft_model(model, peft_config)
# print number of trainable parameters
model.print_trainable_parameters()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,061,890 || all params: 356,423,684 || trainable%: 0.2979


## 模型微调（prompt tuning）

指定优化器和学习率调整策略

In [20]:
import torch
optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)

# Instantiate scheduler
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0.06 * (len(train_dataset) * num_epochs),
    num_training_steps=(len(train_dataset) * num_epochs),
)

打印参与微调的模型参数

In [21]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.shape}")

base_model.classifier.modules_to_save.default.dense.weight: mindtorch.Size([1024, 1024])
base_model.classifier.modules_to_save.default.dense.bias: mindtorch.Size([1024])
base_model.classifier.modules_to_save.default.out_proj.weight: mindtorch.Size([2, 1024])
base_model.classifier.modules_to_save.default.out_proj.bias: mindtorch.Size([2])
prompt_encoder.default.embedding.weight: mindtorch.Size([10, 1024])


In [ ]:
model.npu()  #模型移动到npu侧
device = model.device  # 获取模型所在的设备
device

device(type=npu, index=0)

按照如下步骤定义训练逻辑：

1. 构建正向计算函数
2. 函数变换，获取微分函数
3. 定义训练一个step的逻辑
4. 遍历训练数据集进行模型训练，同时每一个epoch后，遍历验证数据集获取当前的评价指标（accuracy、f1 score）

In [32]:
import mindtorch

for epoch in range(num_epochs):
    # 训练阶段
    model.train()
    train_total_size = len(train_dataloader)
    for step, batch in enumerate(tqdm(train_dataloader, total=train_total_size)):
        
        # 将batch中的所有张量移动到模型所在的设备
        batch_on_device = {}
        for key, value in batch.items():
            if mindtorch.is_tensor(value):
                batch_on_device[key] = value.to(device)
            else:
                batch_on_device[key] = value
                
        # 手动清零梯度，避免调用optimizer.zero_grad()
        for param in model.parameters():
            if param.grad is not None:
                param.grad = None  
             
        outputs = model(**batch_on_device)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

    # 评估阶段
    model.eval()
    eval_total_size = len(eval_dataloader)
    for step, batch in enumerate(tqdm(eval_dataloader, total=eval_total_size)):    
        # 将batch中的所有张量移动到模型所在的设备
        batch_on_device = {}
        for key, value in batch.items():
            if mindtorch.is_tensor(value):
                batch_on_device[key] = value.to(device)
            else:
                batch_on_device[key] = value
        with torch.no_grad():
            outputs = model(**batch_on_device)
        
        predictions = outputs.logits.argmax(dim=-1)
        predictions, references = predictions.asnumpy(), batch["labels"].asnumpy()
        metric.add_batch(
            predictions=predictions,
            references=references,
        )

    eval_metric = metric.compute()
    print(f"epoch {epoch}:", eval_metric)

100%|██████████| 13/13 [00:02<00:00,  5.26it/s]


epoch 0: {'accuracy': 0.6838235294117647, 'f1': 0.8122270742358079}
